# 🇺🇬 Speak Uganda: build an AI language buddy that talks
**Godly Geeks Movement · CoCIS Christian Fellowship · Makerere University · 7 Nov 2026**
Facilitator: **Yiga Gilbert**, Software Engineer, Sunbird AI

<a href="https://colab.research.google.com/github/yigagilbert/Future_of_ai_session/blob/main/notebooks/workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We build **one project** in four steps. By the end, an AI agent will check the weather in Gulu, teach you Acholi phrases and **say them out loud**.

| Step | Topic | What we add |
|---|---|---|
| 1 | **LLM development** | Talk to Gemini through its API |
| 2 | **API integration** | Give it tools: weather, Sunbird AI translation and **text-to-speech** |
| 3 | **MCP servers** | Package the tools so *any* AI app can use them |
| 4 | **Agentic AI** | Give it a goal and let it plan and use the tools itself |

> *"All of them were filled with the Holy Spirit and began to speak in other tongues… each one heard their own language being spoken."* (Acts 2:4, 6)

Full code to take home: **https://github.com/yigagilbert/Future_of_ai_session**

## 0. Setup (2 minutes)

1. Get a **free Gemini API key**: https://aistudio.google.com/apikey
2. In Colab, click the **🔑 key icon** in the left sidebar → *Add new secret* → name it `GEMINI_API_KEY`, paste the key, and turn on *Notebook access*.
3. *(Optional, but it's the fun part)* add `SUNBIRD_API_KEY` the same way for translation and speech in Ugandan languages: https://api.sunbird.ai

If you skip the secrets, the cell below will ask you to paste the key instead.

In [ ]:
!pip install -q google-genai "mcp>=2.3,<3" requests python-dotenv
!git clone -q https://github.com/yigagilbert/Future_of_ai_session.git repo 2>/dev/null || git -C repo pull -q
%cd repo

In [ ]:
import os
from getpass import getpass

def load_key(name, required=True):
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
    except Exception:
        if required or input(f"Do you have a {name}? (y/n) ").lower().startswith("y"):
            os.environ[name] = getpass(f"Paste your {name}: ")
    print(f"{name}: {'✅ set' if os.environ.get(name) else '⚠️ not set'}")

load_key("GEMINI_API_KEY")
load_key("SUNBIRD_API_KEY", required=False)

from google import genai
from google.genai import types

MODEL = "gemini-2.5-flash"  # free tier. Change here if Google renames models.

client = genai.Client(
    # The free tier allows only a few requests per minute: retry instead of crashing.
    http_options=types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=5, initial_delay=2, max_delay=30))
)

# A little helper to play any new audio files our tools create
import glob
from IPython.display import Audio, display
_played = set()
def play_new():
    for f in sorted(glob.glob("output/audio/*")):
        if f not in _played:
            _played.add(f)
            print("🔊", f)
            display(Audio(filename=f))

print("Ready ✅")

---
# Step 1: LLM development 🧠
An LLM is a function: **text in → text out**. We reach it through an **API** with an **API key**.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="In two sentences, what is a large language model?",
)
print(response.text)

### System instructions and temperature
The **system instruction** sets the model's role and style. **Temperature** controls creativity (0 = predictable, higher = more varied).

In [ ]:
config = types.GenerateContentConfig(
    system_instruction=(
        "You are 'Speak Uganda', a cheerful language buddy for university students at Makerere. "
        "You love Uganda's languages and cultures. Keep answers under 100 words."
    ),
    temperature=0.7,
)
response = client.models.generate_content(
    model=MODEL, contents="How many languages are spoken in Uganda, and which are the biggest?", config=config
)
print(response.text)

### Tokens: what you pay for (or what your rate limit counts)

In [ ]:
u = response.usage_metadata
print(f"prompt tokens: {u.prompt_token_count} | answer tokens: {u.candidates_token_count} | total: {u.total_token_count}")

### Chat = re-sending the history
The model has **no memory**. A chat works because the SDK sends the whole conversation every time.

In [ ]:
chat = client.chats.create(model=MODEL, config=config)
for q in ["I'm visiting Gulu next week. Which language is spoken there?", "Teach me how to greet someone in it."]:
    print("You:", q)
    print("AI: ", chat.send_message(q).text, "\n")

### ⚠️ The limitation
Ask for live data or a skill it doesn't have:

In [ ]:
print(client.models.generate_content(
    model=MODEL,
    contents="What's the weather in Gulu tomorrow? And say 'Congratulations on your wedding' in Acholi, out loud please.",
).text)

👉 No live weather, a possibly wrong Acholi guess, and no voice. **Let's give it tools.**

---
# Step 2: API integration (tool / function calling) 🔧
The model **cannot call APIs itself**. Instead:
1. We **describe** our tools (name + description + JSON schema).
2. The model replies: *"please call `get_weather` with `city='Gulu'`"*.
3. **Our code** calls the API and sends the result back.
4. The model writes the final answer with real data.

Our tools are plain Python functions in `speak_uganda/tools.py`:

| Tool | API |
|---|---|
| `get_weather(city, days)` | Open-Meteo (free, no key) |
| `translate_text(text, target_language)` | Sunbird AI |
| `speak_text(text, language)` | Sunbird AI text-to-speech |
| `save_phrasebook(title, content)` | saves a file |

Here's the one that talks:

In [ ]:
import inspect
from speak_uganda.tools import get_weather, translate_text, speak_text, save_phrasebook, ALL_TOOLS
print(inspect.getsource(speak_text))

Plain Python, no AI involved yet:

In [ ]:
print(get_weather("Gulu", days=2))
print(translate_text("Thank you very much", "lug"))

In [ ]:
print(speak_text("Webale nnyo", "lug"))   # 'Thank you very much' in Luganda
play_new()

### Describe the tools to Gemini
This JSON schema is the **contract** between our code and the model. Remember it: MCP uses the same idea.

In [ ]:
LANG = "ISO 639-3 code: lug=Luganda, ach=Acholi, teo=Ateso, lgg=Lugbara, nyn=Runyankole, swa=Swahili"

tool_declarations = [
    types.FunctionDeclaration(
        name="get_weather",
        description="Get the weather forecast for a town or city in Uganda.",
        parameters_json_schema={
            "type": "object",
            "properties": {"city": {"type": "string"}, "days": {"type": "integer", "description": "1-7"}},
            "required": ["city"],
        },
    ),
    types.FunctionDeclaration(
        name="translate_text",
        description="Translate English text into a Ugandan language using Sunbird AI.",
        parameters_json_schema={
            "type": "object",
            "properties": {"text": {"type": "string"}, "target_language": {"type": "string", "description": LANG}},
            "required": ["text", "target_language"],
        },
    ),
    types.FunctionDeclaration(
        name="speak_text",
        description="Speak text aloud in a Ugandan language (Sunbird AI text-to-speech). The text must already be in that language.",
        parameters_json_schema={
            "type": "object",
            "properties": {"text": {"type": "string"}, "language": {"type": "string", "description": LANG}},
            "required": ["text", "language"],
        },
    ),
]
tool_config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=tool_declarations)],
    system_instruction="You are Speak Uganda. Always use tools for weather, translations and speech; never guess. "
                       "To speak a phrase, translate it first, then speak the translation.",
)

# Watch what the model sends back: a REQUEST to call a function, not an answer
r = client.models.generate_content(model=MODEL, contents="Will it rain in Gulu this weekend?", config=tool_config)
print(r.function_calls)

### The tool-calling loop

In [ ]:
def ask(question):
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]
    while True:
        response = client.models.generate_content(model=MODEL, contents=contents, config=tool_config)
        if not response.function_calls:            # no tool requested -> final answer
            return response.text
        contents.append(response.candidates[0].content)   # keep the model's request in history
        parts = []
        for call in response.function_calls:
            print(f"   🔧 {call.name}({dict(call.args)})")
            result = ALL_TOOLS[call.name](**call.args)  # WE run the tool
            print(f"   ↳ {result[:100]}")
            parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
        contents.append(types.Content(role="user", parts=parts))

print(ask("Will it rain in Gulu this weekend?"))

In [ ]:
print(ask("How do I say 'Good morning, my friend' in Acholi? Let me hear it."))
play_new()

### Shortcut: automatic function calling
The SDK can run that loop for you. Pass plain Python functions; it reads their **type hints and docstrings** to build the schema.

In [ ]:
r = client.models.generate_content(
    model=MODEL,
    contents="Compare tomorrow's weather in Jinja and Mbarara. Where should I go for a picnic?",
    config=types.GenerateContentConfig(tools=[get_weather]),
)
print(r.text)

---
# Step 3: MCP servers 🔌
**Problem:** our tools are locked inside *this notebook*, wired to *one* model. What if Claude Desktop, VS Code, Cursor or another app needs them?

**MCP (Model Context Protocol)** is an open standard for exposing tools to *any* AI app. Think of it as **USB-C for AI tools**.

```
 Gemini / Claude / any LLM ◀──▶ MCP client ◀──MCP──▶ speak-uganda server ──▶ Open-Meteo, Sunbird AI
```

### The server: a few lines wrapping the same functions

In [ ]:
%%writefile mcp_server.py
from mcp.server.mcpserver import MCPServer
from speak_uganda import tools

mcp = MCPServer(name="speak-uganda")

@mcp.tool()
def get_weather(city: str, days: int = 3) -> str:
    """Get the weather forecast for a town or city in Uganda (1-7 days)."""
    return tools.get_weather(city, days)

@mcp.tool()
def translate_text(text: str, target_language: str) -> str:
    """Translate English text with Sunbird AI. Codes: lug=Luganda, ach=Acholi, teo=Ateso, lgg=Lugbara, nyn=Runyankole."""
    return tools.translate_text(text, target_language)

@mcp.tool()
def speak_text(text: str, language: str = "lug") -> str:
    """Speak text aloud with Sunbird AI and save the audio. The text must already be in that language."""
    return tools.speak_text(text, language)

@mcp.tool()
def save_phrasebook(title: str, content: str) -> str:
    """Save a finished phrasebook (Markdown) to a file."""
    return tools.save_phrasebook(title, content)

if __name__ == "__main__":
    mcp.run()   # talks MCP over stdin/stdout

### The client: discover the tools, then hand them to Gemini
Note: **this client contains no tool code at all.** It asks the server what it can do.

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command=sys.executable, args=["mcp_server.py"], env=dict(os.environ))
ERRLOG = open("mcp_server.log", "w")   # Colab's stderr can't be handed to a subprocess

async with stdio_client(SERVER, errlog=ERRLOG) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        listed = await session.list_tools()
        for tool in listed.tools:
            print(f"🔌 {tool.name}\n   schema: {tool.input_schema}\n")

In [ ]:
def mcp_tools_to_gemini(mcp_tools):
    # MCP tools already carry a JSON schema, so conversion is one line each
    return types.Tool(function_declarations=[
        types.FunctionDeclaration(name=t.name, description=t.description or "", parameters_json_schema=t.input_schema)
        for t in mcp_tools
    ])

async def call_mcp_tool(session, name, args):
    result = await session.call_tool(name, args)
    return "\n".join(c.text for c in result.content if getattr(c, "text", None))

async def ask_via_mcp(question):
    async with stdio_client(SERVER, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            config = types.GenerateContentConfig(tools=[mcp_tools_to_gemini((await session.list_tools()).tools)])
            contents = [types.Content(role="user", parts=[types.Part(text=question)])]
            while True:
                response = await client.aio.models.generate_content(model=MODEL, contents=contents, config=config)
                if not response.function_calls:
                    return response.text
                contents.append(response.candidates[0].content)
                parts = []
                for call in response.function_calls:
                    print(f"   🔌 MCP → {call.name}({dict(call.args)})")
                    result = await call_mcp_tool(session, call.name, dict(call.args))
                    parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
                contents.append(types.Content(role="user", parts=parts))

print(await ask_via_mcp("What's the weather in Jinja tomorrow? Then teach me 'good morning' in Luganda and let me hear it."))
play_new()

💡 **The payoff:** plug the *same* `step3_mcp_server.py` into Claude Desktop and **Claude can speak Luganda**. See the repo README for the config.

---
# Step 4: Agentic AI 🤖
A **chatbot** answers a question. An **agent** pursues a **goal**:

```
     ┌──────────────► THINK (plan the next step)
     │                    │
  OBSERVE ◄── ACT (call a tool)
     │
     └── repeat until the goal is done (or a guardrail stops it)
```

Same loop as Step 3. What changes is the **job**: an open-ended goal that needs several tools in sequence, plus instructions that make the model plan and check its own work.

In [ ]:
MAX_STEPS = 15   # guardrail: never loop forever (and protect your free quota)

AGENT_INSTRUCTIONS = '''
You are Speak Uganda, an agent that prepares short, practical language lessons for
university students travelling around Uganda.
1. Start by writing a short numbered PLAN.
2. Check the weather for the destination and let it shape your tips.
3. Use translate_text for EVERY phrase. Never write a Ugandan-language translation from memory.
4. Use speak_text on each translated phrase, in the same language. If a tool fails, adapt instead of giving up.
5. End with a short blessing for the journey, also translated.
6. Save everything with save_phrasebook, then reply with a summary starting "DONE:".
Be warm and encouraging. Keep it short.
'''

async def run_agent(goal):
    async with stdio_client(SERVER, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            config = types.GenerateContentConfig(
                system_instruction=AGENT_INSTRUCTIONS,
                tools=[mcp_tools_to_gemini((await session.list_tools()).tools)],
            )
            contents = [types.Content(role="user", parts=[types.Part(text=goal)])]
            for step in range(1, MAX_STEPS + 1):
                response = await client.aio.models.generate_content(model=MODEL, contents=contents, config=config)
                contents.append(response.candidates[0].content)
                thought = "".join(p.text or "" for p in response.candidates[0].content.parts).strip()
                if thought:
                    print(f"\n🧠 [step {step}] THINK:\n{thought}")
                if not response.function_calls:
                    return thought
                parts = []
                for call in response.function_calls:
                    args = dict(call.args)
                    print(f"🔧 [step {step}] ACT: {call.name}({ {k: str(v)[:60] for k, v in args.items()} })")
                    result = await call_mcp_tool(session, call.name, args)
                    print(f"👀 [step {step}] OBSERVE: {result[:120]}")
                    parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
                contents.append(types.Content(role="user", parts=parts))
            return "Stopped: step limit reached (guardrail)."

In [ ]:
goal = (
    "I'm a Makerere student travelling to Gulu this weekend for a friend's wedding. "
    "Check the weather, teach me 4 useful Acholi phrases for the trip and the wedding, "
    "let me hear each one, and save a phrasebook."
)
final = await run_agent(goal)

In [ ]:
# Read the phrasebook and listen to every phrase
for f in sorted(glob.glob("output/*.md")):
    print("📄", f, "\n")
    print(open(f).read())
play_new()

### Try your own goal
Ask the room! Some ideas: exam week in Mbarara (Runyankole), a football match in Soroti (Ateso), a visit to grandma in Arua (Lugbara).

In [ ]:
await run_agent("I'm visiting my grandmother in Arua. Check the weather and teach me 3 respectful Lugbara greetings I can say to her elders. Let me hear them.")
play_new()

---
# 🎓 Wrap-up

| Concept | One-liner |
|---|---|
| **LLM** | Text in → text out, through an API. No memory, no live data. |
| **Tool calling** | The model *asks*; your code *acts*. |
| **MCP** | A standard plug so any AI app can use your tools. |
| **Agent** | A loop: think → act → observe, until the goal is met. Add guardrails. |

### Take-home challenges
1. 🟢 Add a tool `get_proverb(language)` that returns a proverb from a Python list, then translate and speak it.
2. 🟡 Add **speech-to-text**: record yourself speaking Luganda and let the agent check your pronunciation (Sunbird `/tasks/audio/transcriptions`).
3. 🟡 Connect `step3_mcp_server.py` to Claude Desktop or VS Code (see the README).
4. 🔴 Build a "language of the week" agent for the fellowship: one phrase, one verse and one audio clip, every Monday.

📦 **Full project:** https://github.com/yigagilbert/Future_of_ai_session

*"Whatever you do, work at it with all your heart, as working for the Lord."* (Colossians 3:23)